# Deploy The Legacy to SageMaker

Interactive walkthrough for deploying the finetuned Llama 3.2 1B LoRA model to a SageMaker real-time endpoint and verifying it works.

**Cost warning:** the endpoint runs on `ml.g5.xlarge` at ~$1.41/hr, billed continuously until deleted.

| Duration | Cost |
|---|---|
| 1 hour  | ~$1.41 |
| 1 day   | ~$34 |
| 1 week  | ~$237 |

**Don't forget the teardown cell at the bottom when you're done.**

---

## Prerequisites

Before running this notebook, you should have:

- AWS account with SageMaker access
- `aws configure` completed (access key, secret, default region)
- A SageMaker execution role — IAM → Roles, attach `AmazonSageMakerFullAccess`. You'll paste its ARN in the config cell below.
- HuggingFace token with **write** access, and `huggingface-cli login` run in the terminal
- Python deps installed:
  ```
  pip install boto3 sagemaker transformers peft torch huggingface_hub accelerate
  ```

Full walkthrough with troubleshooting: [notes/development/sagemaker-deployment.md](../notes/development/sagemaker-deployment.md)

## 1. Configuration

Edit the values below to match your setup. The defaults work for most cases.

In [ ]:
# === REQUIRED: paste your SageMaker execution role ARN here ===
ROLE_ARN = "arn:aws:iam::ACCOUNT_ID:role/YOUR-SAGEMAKER-ROLE-NAME"

# === Other config (change only if you know what you're doing) ===
AWS_REGION = None  # None = use your default AWS region; override with e.g. "us-east-1"
ENDPOINT_NAME = "the-legacy-llm"
INSTANCE_TYPE = "ml.g5.xlarge"
HF_MERGED_REPO = "malhl/the-legacy-lora-merged"
HF_ADAPTER_REPO = "malhl/the-legacy-lora"

print(f"Will deploy {HF_MERGED_REPO} to endpoint '{ENDPOINT_NAME}' on {INSTANCE_TYPE}")
if "ACCOUNT_ID" in ROLE_ARN:
    print("\n[WARN] ROLE_ARN still has placeholder values. Edit the cell above before continuing.")

## 2. Verify AWS credentials

Quick check that boto3 is installed and credentials are configured. If this fails, run `aws configure` first.

In [ ]:
import boto3

boto_sess = boto3.Session(region_name=AWS_REGION) if AWS_REGION else boto3.Session()
sts = boto_sess.client("sts")
identity = sts.get_caller_identity()

print(f"Account:  {identity['Account']}")
print(f"User/role: {identity['Arn']}")
print(f"Region:   {boto_sess.region_name}")

## 3. Merge LoRA adapter and push merged model to HuggingFace

SageMaker's TGI container pulls the model from HuggingFace, so we need the fully merged model on HF Hub (not just the adapter).

This shells out to `scripts/merge_and_convert.py --push-hf-repo ... --skip-gguf`.

**Skip this cell if `malhl/the-legacy-lora-merged` already exists on HF** (check [huggingface.co/malhl/the-legacy-lora-merged](https://huggingface.co/malhl/the-legacy-lora-merged)).

Takes ~3-5 minutes (mostly the upload).

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
script = REPO_ROOT / "scripts" / "merge_and_convert.py"

result = subprocess.run(
    [
        sys.executable, str(script),
        "--push-hf-repo", HF_MERGED_REPO,
        "--adapter-repo", HF_ADAPTER_REPO,
        "--skip-gguf",
    ],
    cwd=REPO_ROOT,
)
if result.returncode != 0:
    raise RuntimeError("Merge and push failed — check output above")
print(f"\nMerged model pushed to https://huggingface.co/{HF_MERGED_REPO}")

## 4. Check if the endpoint already exists

If you've run this notebook before, the endpoint may still be up (and still billing). Check first.

In [ ]:
sm = boto_sess.client("sagemaker")

try:
    existing = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    print(f"Endpoint '{ENDPOINT_NAME}' already exists with status: {existing['EndpointStatus']}")
    print(f"Created: {existing['CreationTime']}")
    print("\nTo delete it first, run the teardown cell at the bottom of this notebook.")
    print("To use the existing endpoint, skip the 'Create endpoint' cell below.")
except sm.exceptions.ClientError as e:
    if "Could not find endpoint" in str(e):
        print(f"No existing endpoint named '{ENDPOINT_NAME}'. Safe to create one.")
    else:
        raise

## 5. Create the endpoint

This takes 5-10 minutes. The container has to download ~2.5GB from HuggingFace on cold start.

**Billing starts immediately** at ~$1.41/hour.

In [ ]:
import sagemaker
from sagemaker.huggingface import HuggingFaceModel, get_huggingface_llm_image_uri

sm_sess = sagemaker.Session(boto_session=boto_sess)

image_uri = get_huggingface_llm_image_uri("huggingface", version="3.2.3")

model = HuggingFaceModel(
    image_uri=image_uri,
    role=ROLE_ARN,
    env={
        "HF_MODEL_ID": HF_MERGED_REPO,
        "HF_TASK": "text-generation",
        "MAX_INPUT_LENGTH": "1024",
        "MAX_TOTAL_TOKENS": "2048",
        "SM_NUM_GPUS": "1",
    },
    sagemaker_session=sm_sess,
)

print(f"Deploying {HF_MERGED_REPO} to {ENDPOINT_NAME} ({INSTANCE_TYPE})...")
print("This takes 5-10 minutes. The cell output will show progress.\n")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
    container_startup_health_check_timeout=600,
)

print(f"\n✓ Endpoint created: {ENDPOINT_NAME}")

## 6. Verify the endpoint is InService

The deploy() call blocks until the endpoint is ready, but this is a quick sanity check.

In [ ]:
resp = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
print(f"Status:           {resp['EndpointStatus']}")
print(f"Created:          {resp['CreationTime']}")
print(f"Last modified:    {resp['LastModifiedTime']}")
if resp.get("FailureReason"):
    print(f"\n[FAILURE] {resp['FailureReason']}")

## 7. Test the endpoint

Send a Legacy-specific prompt and check the response is domain-aware (not a generic Llama base model answer).

In [ ]:
import json

runtime = boto_sess.client("sagemaker-runtime")

SYSTEM = (
    "You are The Legacy, an expert AI assistant for Magic: The Gathering "
    "Legacy format."
)

def format_prompt(system: str, user: str) -> str:
    return (
        "<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n\n{system}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

def ask(prompt: str) -> str:
    payload = {
        "inputs": format_prompt(SYSTEM, prompt),
        "parameters": {
            "max_new_tokens": 400,
            "temperature": 0.3,
            "top_p": 0.9,
            "do_sample": True,
        },
    }
    resp = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps(payload),
    )
    result = json.loads(resp["Body"].read().decode())
    if isinstance(result, list):
        result = result[0]
    return result.get("generated_text", "")


question = "What is the most played deck in Legacy right now?"
print(f"Q: {question}\n")
print(f"A: {ask(question)}")

### Expected response

Should mention **Dimir Tempo** (ideally with 14.6% meta share). If it's a generic hedge about not knowing current metas, the pushed model is probably not the finetuned one — re-check step 3.

Try a few more to confirm the Round 2 wins are all working:

In [ ]:
tests = [
    ("What is a budget replacement for Underground Sea?",
     "Should say Watery Grave (shockland), NOT Mox Diamond"),
    ("Is Counterspell good in Legacy? What does it cost?",
     "Should say UU (2 mana), NOT 1UU or 'mana value 2 or less'"),
    ("What are Orcish Bowmasters' stats?",
     "Should say 1/1 for 1B with Flash, NOT 1GG or trample"),
]

for q, expected in tests:
    print(f"Q: {q}")
    print(f"   (expected: {expected})")
    print(f"A: {ask(q)[:400]}...\n")

## 8. Run the full smoke test

Shells out to `scripts/test_deployment.py --sagemaker`, which runs 5 Legacy-specific prompts with expected-vs-reject pattern matching. Confirms the Round 2 wins held up end-to-end.

In [ ]:
cmd = [
    sys.executable, str(REPO_ROOT / "scripts" / "test_deployment.py"),
    "--sagemaker",
    "--sagemaker-endpoint", ENDPOINT_NAME,
]
if AWS_REGION:
    cmd.extend(["--region", AWS_REGION])

subprocess.run(cmd, cwd=REPO_ROOT)

## 9. (Optional) Point the FastAPI server at the endpoint

Copy these to a terminal to run the local API against the SageMaker-deployed model:

**Bash:**
```bash
export INFERENCE_BACKEND=sagemaker
export SAGEMAKER_ENDPOINT=the-legacy-llm
export AWS_REGION=us-east-1  # or wherever you deployed
uvicorn src.server:app --reload --port 8000
```

**PowerShell:**
```powershell
$env:INFERENCE_BACKEND = "sagemaker"
$env:SAGEMAKER_ENDPOINT = "the-legacy-llm"
$env:AWS_REGION = "us-east-1"
uvicorn src.server:app --reload --port 8000
```

Then hit [http://localhost:8000/health](http://localhost:8000/health) to confirm it's connected.

---

## 10. TEARDOWN — run this when you're done!

Deletes the endpoint, endpoint config, and associated model objects. **Billing stops once the endpoint deletion completes** (takes ~30 seconds).

If you skip this, the endpoint keeps running at ~$1.41/hr until you manually clean it up.

In [ ]:
# Show current cost estimate before teardown
from datetime import datetime, timezone

try:
    resp = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    hours = (datetime.now(timezone.utc) - resp["CreationTime"]).total_seconds() / 3600
    print(f"Endpoint ran for ~{hours:.2f} hours → estimated cost ${hours * 1.41:.2f}")
except sm.exceptions.ClientError:
    print("Endpoint already deleted — nothing to tear down.")

In [ ]:
# Actually delete it
deleted_something = False

try:
    sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print(f"✓ Endpoint deleted: {ENDPOINT_NAME}")
    deleted_something = True
except sm.exceptions.ClientError as e:
    if "Could not find endpoint" in str(e):
        print(f"  (Endpoint {ENDPOINT_NAME} already gone)")
    else:
        print(f"[WARN] Failed to delete endpoint: {e}")

try:
    sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print(f"✓ Endpoint config deleted")
    deleted_something = True
except sm.exceptions.ClientError:
    pass

try:
    resp = sm.list_models(NameContains="huggingface-pytorch-tgi")
    for m in resp.get("Models", []):
        sm.delete_model(ModelName=m["ModelName"])
        print(f"✓ Model object deleted: {m['ModelName']}")
        deleted_something = True
except Exception as e:
    print(f"[WARN] Could not clean up model objects: {e}")

if deleted_something:
    print("\nAll resources cleaned up. Billing has stopped.")